# Random Forest Feature Detection

## Setup 

First we need our tools. pandas gives us the DataFrame, very similar 
to R's DataFrames. The DataFrame is a structure that allows us to
work with our data more easily. It has nice features for slicing 
and transformation of data, and easy ways to do basic statistics. 

numpy has some very handy functions that work on DataFrames.

## Dataset

We are using a dataset about the wine quality dataset, archived at UCI's Machine Learning Repository (http://archive.ics.uci.edu/ml/index.php).

In [ ]:
import pandas as pd
import numpy as np

Now we'll load our data. pandas makes it easy!

In [ ]:
# red wine quality data, packed in a DataFrame
red_df = pd.read_csv('winequality-red.csv',sep=';',header=0, index_col=False) 

# white wine quality data, packed in a DataFrame
white_df = pd.read_csv('winequality-white.csv',sep=';',header=0,index_col=False) 

# rose? other fruit wines? plum wine? :(

Like in R, there is a .describe() method that gives basic statistics for every column in the dataset.

In [ ]:
# for red wines
red_df.describe()

In [ ]:
# for white wines 
white_df.describe()

Sometimes it is easier to understand the data visually. A histogram of 
the white wine quality *data citric* acid samples is shown below. You 
can of course visualize other columns' data or other datasets. Just 
replace the DataFrame and column name below.

In [ ]:
import matplotlib.pyplot as plt

def extract_col(df,col_name):
    return list(df[col_name])

col = extract_col(white_df,'citric acid') # can replace with another dataframe or column
plt.hist(col)

#TODO: add axes and such to set a good example

plt.show()

## Detecting Features

Let us try out a some elementary machine learning models. These models 
are not always for prediction. They are also useful to find what 
features are most predictive of a variable of interest. Depending 
on the classifier you use, you may need to transform the data 
pertaining to that variable.

### Data Preparation

Let us assume we want to study what features are most correlated 
with pH. pH of course is real-valued, and continuous. The classifiers 
we want to use usually need labeled or integer data. Hence, we will 
transform the pH data, assigning wines with pH higher than average 
as ``hi`` (more basic or alkaline) and wines with pH lower than 
average as ``lo`` (more acidic). 

In [ ]:
# refresh to make Jupyter happy
red_df = pd.read_csv('winequality-red.csv',sep=';',header=0, index_col=False) 
white_df = pd.read_csv('winequality-white.csv',sep=';',header=0,index_col=False) 

#TODO: data cleansing functions here, e.g. replacement of NaN

# if the variable you want to predict is continuous, you can map ranges of values
# to integer/binary/string labels

# for example, map the pH data to 'hi' and 'lo' if a pH value is more than or
# less than the mean pH, respectively
M = np.mean(list(red_df['pH'])) # expect inelegant code in these mappings
Lf = lambda p: int(p < M)*'lo' + int(p >= M)*'hi' # some C-style hackery 

# create the new classifiable variable
red_df['pH-hi-lo'] = map(Lf,list(red_df['pH']))

# and remove the predecessor
del red_df['pH']

Now we specify which dataset and variable you want to predict by 
assigning vlues to ``SELECTED_DF`` and ``TARGET_VAR``, respectively. 

We like to keep a parameter file where we specify data sources 
and such. This lets me create generic analytics code that is easy 
to reuse. 

After we have specified what dataset we want to study, we split 
the training and test datasets. We then scale (normalize) the data, 
which makes most classifiers run better. 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import metrics

# make selections here without digging in code
SELECTED_DF = red_df # selected dataset
TARGET_VAR = 'pH-hi-lo' # the predicted variable

# generate nameless data structures
df = SELECTED_DF
target = np.array(df[TARGET_VAR]).ravel()
del df[TARGET_VAR] # no cheating

#TODO: data cleansing function calls here

# split datasets for training and testing
X_train, X_test, y_train, y_test = train_test_split(df,target,test_size=0.2)

# set up the scaler
scaler = StandardScaler()
scaler.fit(X_train)

# apply the scaler
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

Now we pick a classifier. As you can see, there are many to try 
out, and even more in scikit-learn's documentation and many 
examples and tutorials. Random Forests are data science workhorses. 
They are the go-to method for most data scientists. Be careful 
relying on them though--they tend to overfit. We try to avoid 
overfitting by separating the training and test datasets. 

## Random Forest

In [ ]:
# pick a classifier

from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor,ExtraTreeClassifier,ExtraTreeRegressor
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier

clf = RandomForestClassifier()

Now we will test it out with the default parameters.

Note that this code is boilerplate. You can use it interchangeably for most scikit-learn models. 

In [ ]:
# test it out

model = clf.fit(X_train,y_train)
pred = clf.predict(X_test)
conf_matrix = metrics.confusion_matrix(y_test,pred)

var_score = clf.score(X_test,y_test)

# the results
importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]

Now output the results. For Random Forests, we get a feature ranking. 
Relative importances usually exponentially decay. The first few 
highly-ranked features are usually the most important. 

In [ ]:
# for the sake of clarity
num_features = X_train.shape[1]
features = map(lambda x: df.columns[x],indices)
feature_importances = map(lambda x: importances[x],indices)

print 'Feature ranking:\n'

for i in range(num_features):
    feature_name = features[i]
    feature_importance = feature_importances[i]
    print '%s%f' % (feature_name.ljust(30), feature_importance)

Sometimes it's easier to visualize. We'll use a bar chart. 

In [ ]:
plt.clf()
plt.bar(range(num_features),feature_importances)
plt.xticks(range(num_features),features,rotation=90)
plt.ylabel('relative importance (a.u.)')
plt.title('Relative importances of most predictive features')
plt.show()

In [ ]:
import dask.dataframe as dd

red_df = dd.read_csv('winequality-red.csv',sep=';',header=0) 
white_df = dd.read_csv('winequality-white.csv',sep=';',header=0) 

## Acknowledgement

This notebook was developed by Juliette Zerick and Gregor von Laszewski